# Ачча! — озвучка фраз (Indic Parler-TTS)

Блокнот озвучивает фразы приложения «Ачча!» открытой моделью **Indic Parler-TTS** (AI4Bharat, лицензия Apache 2.0) и сохраняет mp3 в ваш Google Drive.

**Перед запуском:** меню **Среда выполнения → Сменить среду выполнения → Аппаратный ускоритель: T4 GPU → Сохранить**.

Дальше запускайте ячейки по порядку (кнопка ▶ слева от ячейки). Шаг 5 — пробное прослушивание: если голос не понравился, поменяйте имя голоса на шаге 2 и запустите шаги 2 и 5 ещё раз. Шаг 6 озвучивает всё (10–20 минут).

## 1. Установка (2–3 минуты)

In [5]:
!pip -q install --upgrade protobuf
!pip -q install git+https://github.com/huggingface/parler-tts.git soundfile
!apt -qq install -y ffmpeg > /dev/null
!pip -q install -U "protobuf>=5.28"
print('Готово')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-dataproc 5.30.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.84.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
proto-plus 1.29.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 4.25.9 which is incompatible.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-bigquery-storage 2.41.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which

## 2. Настройки

Для английского создатели модели рекомендуют голоса **Thoma** (мужской) и **Mary** (женский); полный список из 21 голоса — на странице модели huggingface.co/ai4bharat/indic-parler-tts.
- `LOCAL_VOICE` — голос для реплик индийцев (индийский английский)
- `TOURIST_VOICE` — голос для реплик туриста
- `ONLY_LOCAL = True` — озвучить только реплики индийцев (туристу останется голос телефона)

In [6]:
PHRASES_URL = 'https://achha-india.github.io/app/phrases.json'

LOCAL_VOICE = 'Thoma'
TOURIST_VOICE = 'Mary'
ONLY_LOCAL = False

LOCAL_STYLE = "{name} speaks in a clear Indian English accent at a moderate pace, with a friendly tone. The recording is very clear and close, with no background noise."
TOURIST_STYLE = "{name} speaks slowly and clearly with a neutral accent, in a calm tone. The recording is very clear and close, with no background noise."

OUT_DIR = '/content/drive/MyDrive/Achha/audio'
print('Настройки сохранены')

Настройки сохранены


## 3. Загрузка модели (3–5 минут)

Если появится ошибка про доступ (401 / gated), откройте страницу модели huggingface.co/ai4bharat/indic-parler-tts, войдите в бесплатный аккаунт Hugging Face, нажмите «Agree and access» и добавьте токен в Colab: значок ключа слева → `HF_TOKEN`.

In [7]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', device, '(если cpu — включите T4 GPU, см. начало)')
MODEL_ID = 'ai4bharat/indic-parler-tts'
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
SR = model.config.sampling_rate
print('Модель загружена')

Устройство: cuda (если cpu — включите T4 GPU, см. начало)


  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

  "_name_or_path": "ylacombe/dac_44khz",
  "architectures": [
    "DacModel"
  ],
  "codebook_dim": 8,
  "codebook_loss_weight": 1.0,
  "codebook_size": 1024,
  "commitment_loss_weight": 0.25,
  "decoder_hidden_si

Модель загружена


## 4. Фразы с сайта

In [8]:
import json, urllib.request
data = json.load(urllib.request.urlopen(PHRASES_URL))
lines = [l for st in data['stages'] for s in st['situations'] for l in s['lines']]
todo = [l for l in lines if l['role'] == 'local' or not ONLY_LOCAL]
print('Всего реплик:', len(lines), '| к озвучке:', len(todo))

import soundfile as sf, numpy as np, subprocess, os

def synth(text, role, seed=7):
    name = LOCAL_VOICE if role == 'local' else TOURIST_VOICE
    style = (LOCAL_STYLE if role == 'local' else TOURIST_STYLE).format(name=name)
    torch.manual_seed(seed)
    d = description_tokenizer(style, return_tensors='pt').to(device)
    p = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        g = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                           prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
    a = g.cpu().numpy().squeeze().astype(np.float32)
    # обрезаем тишину по краям
    nz = np.where(np.abs(a) > 0.01)[0]
    if len(nz): a = a[max(0, nz[0] - int(0.05 * SR)): nz[-1] + int(0.15 * SR)]
    return a

def to_mp3(audio, path):
    wav = path.replace('.mp3', '.wav')
    sf.write(wav, audio, SR)
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', wav, '-ac', '1', '-b:a', '64k', path], check=True)
    os.remove(wav)
print('Готово')

Всего реплик: 57 | к озвучке: 57
Готово


## 5. Пробное прослушивание

In [9]:
from IPython.display import Audio, display
samples = [l for l in todo if l['id'] in ('market-02', 'road-13', 'market-01')] or todo[:3]
for l in samples:
    print(l['id'], '·', 'индиец' if l['role'] == 'local' else 'турист', '·', l['en'])
    display(Audio(synth(l['en'], l['role']), rate=SR))

road-13 · индиец · Which side?


market-01 · турист · How much is this?


market-02 · индиец · Five hundred only.


## 6. Озвучить всё и сохранить в Google Drive

Colab попросит доступ к Google Drive — разрешите. Файлы появятся в папке **Achha/audio**, а рядом архив **achha-audio.zip**. Этот архив пришлите в чат.

In [10]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
done, failed = 0, []
for i, l in enumerate(todo, 1):
    path = f"{OUT_DIR}/{l['id']}.mp3"
    if os.path.exists(path):
        done += 1; continue
    try:
        to_mp3(synth(l['en'], l['role']), path)
        done += 1
        print(f"[{i}/{len(todo)}] {l['id']} ✓")
    except Exception as e:
        failed.append(l['id']); print(f"[{i}/{len(todo)}] {l['id']} ошибка: {e}")
import shutil
zip_path = shutil.make_archive('/content/drive/MyDrive/Achha/achha-audio', 'zip', OUT_DIR)
print('\nГотово:', done, '| ошибки:', failed or 'нет')
print('Архив:', zip_path)
from google.colab import files
files.download(zip_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[1/57] air-01 ✓
[2/57] air-02 ✓
[3/57] air-03 ✓
[4/57] air-04 ✓
[5/57] air-05 ✓
[6/57] air-06 ✓
[7/57] air-07 ✓
[8/57] air-08 ✓
[9/57] air-09 ✓
[10/57] air-10 ✓
[11/57] air-11 ✓
[12/57] air-12 ✓
[13/57] air-13 ✓
[14/57] air-14 ✓
[15/57] road-01 ✓
[16/57] road-02 ✓
[17/57] road-03 ✓
[18/57] road-04 ✓
[19/57] road-05 ✓
[20/57] road-06 ✓
[21/57] road-07 ✓
[22/57] road-08 ✓
[23/57] road-09 ✓
[24/57] road-10 ✓
[25/57] road-11 ✓
[26/57] road-13 ✓
[27/57] road-14 ✓
[28/57] road-15 ✓
[29/57] road-16 ✓
[30/57] road-17 ✓
[31/57] road-18 ✓
[32/57] market-01 ✓
[33/57] market-02 ✓
[34/57] market-03 ✓
[35/57] market-04 ✓
[36/57] market-05 ✓
[37/57] market-06 ✓
[38/57] market-07 ✓
[39/57] market-08 ✓
[40/57] market-09 ✓
[41/57] market-10 ✓
[42/57] market-11 ✓
[43/57] market-12 ✓
[44/57] market-13 ✓
[45/57] market-14 ✓
[46/57] market-15 ✓
[47/57] market-16 ✓
[48/57] market-1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Переозвучить отдельные фразы:** удалите их mp3 из папки Achha/audio на Drive и запустите шаг 6 ещё раз — остальные файлы он пропустит. Можно также поменять `seed` в функции `synth` — интонация станет немного другой.